In [3]:
import pandas as pd
import numpy as np
import re
import glob

In [4]:
df = pd.read_csv("../raw_data/amazon_india_2024.csv")

In [3]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2024_00000001,2024-01-03,CUST_2024_00003447,PROD_000387,OnePlus OnePlus 6 64GB White,Electronics,Smartphones,OnePlus,39348.44,0.00,...,False,NaN,NaN,Delivered,1,2024,1,0.24,True,3.4
1,TXN_2024_00000002,2024-01-09,CUST_2018_00008611,PROD_001357,Nothing Phone (2a) Plus 64GB Blue,Electronics,Smartphones,Nothing,"18,694.97",0.00,...,False,NaN,5.0,Delivered,1,2024,1,0.15,True,4.1
2,TXN_2024_00000003,2024-01-09,CUST_2024_00002076,PROD_001899,Xiaomi Watch Premium,Electronics,Smart Watch,Xiaomi,59276.71,17.19,...,False,NaN,NaN,Delivered,1,2024,1,0.06,TRUE,4.2
3,TXN_2024_00000004,2024-01-09,CUST_2024_00014530,PROD_000071,Xiaomi Redmi 2 32GB Black,Electronics,Smartphones,Xiaomi,32605.39,0.00,...,False,NaN,5.0,Delivered,1,2024,1,0.20,True,3.7
4,TXN_2024_00000005,22-01-2024,CUST_2022_00043625,PROD_001144,OnePlus OnePlus 11R 256GB White,Electronics,Smartphones,OnePlus,72805.3,18.89,...,True,Republic Day Sale,4.5,Delivered,1,2024,1,0.21,True,3.6


In [4]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(9728), 121605)

In [5]:
df["delivery_charges"].describe()

count    111877.000000
mean          0.001073
std           0.207132
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          40.000000
Name: delivery_charges, dtype: float64

In [6]:
df.shape

(121605, 34)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121605 entries, 0 to 121604
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          121605 non-null  object 
 1   order_date              121605 non-null  object 
 2   customer_id             121605 non-null  object 
 3   product_id              121605 non-null  object 
 4   product_name            121605 non-null  object 
 5   category                121605 non-null  object 
 6   subcategory             121605 non-null  object 
 7   brand                   121605 non-null  object 
 8   original_price_inr      121605 non-null  object 
 9   discount_percent        121605 non-null  float64
 10  discounted_price_inr    121605 non-null  float64
 11  quantity                121605 non-null  int64  
 12  subtotal_inr            121605 non-null  float64
 13  delivery_charges        111877 non-null  float64
 14  final_amount_inr    

In [8]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_city', 'customer_state', 'customer_tier',
       'customer_spending_tier', 'customer_age_group', 'payment_method',
       'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale',
       'festival_name', 'customer_rating', 'return_status', 'order_month',
       'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [9]:
df["order_date"].head(20)

0     2024-01-03
1     2024-01-09
2     2024-01-09
3     2024-01-09
4     22-01-2024
5     2024-01-25
6     2024-01-12
7     2024-01-12
8     2024-01-25
9     08-01-2024
10    2024-01-18
11    2024-01-31
12    2024-01-16
13    2024-01-14
14    2024-01-06
15    2024-01-28
16    2024-01-26
17    2024-01-14
18    2024-01-23
19    2024-01-18
Name: order_date, dtype: object

In [10]:
df["order_date"] = (
    df["order_date"]
    .str.replace(" ", "", regex=False)
    .str.replace("/", "-", regex=False)
)

parts = df["order_date"].str.split("-", expand=True)

year_last = parts[2].str.len() == 4

df.loc[year_last, "order_date"] = (
    parts[2] + "-" + parts[0] + "-" + parts[1]
)

parts = df["order_date"].str.split("-", expand=True)

mask = parts[1].astype(int) > 12

df.loc[mask, "order_date"] = (
    parts[0] + "-" + parts[2] + "-" + parts[1]
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [11]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [12]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [13]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [14]:
df["original_price_inr"].unique()[:20]

array([39348.44, 18694.97, 59276.71, 32605.39, 72805.3 , 58371.09,
       13588.99, 25059.59, 20009.53, 60611.3 , 47255.83, 24056.88,
       51678.08, 51391.77, 20030.19,      nan, 21555.05, 47090.5 ,
       42799.62, 25329.39])

In [15]:
mask = df["original_price_inr"].isna()

df.loc[mask, "original_price_inr"] = np.where(
    df.loc[mask, "discount_percent"] == 0,
    
    # Case 1: no discount
    df.loc[mask, "discounted_price_inr"],
    
    # Case 2: discount present
    df.loc[mask, "discounted_price_inr"] / (1 - df.loc[mask, "discount_percent"] / 100)
)

In [16]:
df["original_price_inr"].dtypes

dtype('float64')

In [17]:
df["original_price_inr"].isna().sum()

np.int64(0)

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [18]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [19]:
df["customer_rating"].describe()

count    84925.000000
mean         4.305452
std          0.574495
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [20]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    27920
5.0    21583
4.0    21393
3.5     8853
3.0     5176
Name: count, dtype: int64

In [21]:
df["customer_rating"].isna().sum()

np.int64(36680)

In [22]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    36680
4.5    27920
5.0    21583
4.0    21393
3.5     8853
3.0     5176
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [23]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [24]:
df["customer_city"].unique()

array(['chennai', 'mumbai', 'surat', 'ahmedabad', 'delhi', 'bhubaneswar',
       'moradabad', 'indore', 'chandigarh', 'pune', 'jaipur', 'aligarh',
       'varanasi', 'vadodara', 'hyderabad', 'nagpur', 'coimbatore',
       'bangalore', 'visakhapatnam', 'kochi', 'allahabad', 'meerut',
       'kolkata', 'lucknow', 'saharanpur', 'bareilly', 'gorakhpur',
       'ludhiana', 'patna', 'kanpur', 'new delhi', 'bengaluru',
       'bengalore', 'chenai', 'mumba', 'delhi ncr', 'bombay', 'madras',
       'banglore', 'calcutta'], dtype=object)

In [25]:
city_map = {
    "new delhi": "delhi",
    "delhi ncr": "delhi",

    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "bengaluru": "bangalore",
    "bengalore": "bangalore",
    "banglore": "bangalore"
}

In [26]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [27]:
df["customer_city"] = df["customer_city"].str.title()

In [28]:
df["customer_city"].value_counts().head(20)

customer_city
Mumbai           12619
Delhi            10968
Bangalore         9030
Pune              8076
Chennai           7517
Ahmedabad         6468
Kolkata           5717
Jaipur            4980
Surat             4926
Nagpur            4402
Kanpur            4185
Lucknow           4155
Indore            4093
Hyderabad         3867
Coimbatore        3506
Kochi             3318
Visakhapatnam     2905
Patna             2767
Vadodara          2668
Bhubaneswar       2568
Name: count, dtype: int64

In [29]:
df["customer_city"].unique()

array(['Chennai', 'Mumbai', 'Surat', 'Ahmedabad', 'Delhi', 'Bhubaneswar',
       'Moradabad', 'Indore', 'Chandigarh', 'Pune', 'Jaipur', 'Aligarh',
       'Varanasi', 'Vadodara', 'Hyderabad', 'Nagpur', 'Coimbatore',
       'Bangalore', 'Visakhapatnam', 'Kochi', 'Allahabad', 'Meerut',
       'Kolkata', 'Lucknow', 'Saharanpur', 'Bareilly', 'Gorakhpur',
       'Ludhiana', 'Patna', 'Kanpur'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [30]:
bool_candidates = []

bool_values = {"true","false","yes","no","y","n","1","0"}

for col in df.columns:
    vals = set(df[col].astype(str).str.lower().dropna().unique())
    
    if vals & bool_values:
        bool_candidates.append(col)

bool_candidates

['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [31]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [32]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
True             True               False               41841
False            True               False               27611
True             True               True                18431
False            True               True                12486
True             False              False                9032
False            False              False                5626
True             False              True                 4040
False            False              True                 2538
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [5]:
df["category"].value_counts().head(20)

category
Electronics                  121525
Electronics & Accessories        24
Electronic                       23
ELECTRONICS                      17
Electronicss                     16
Name: count, dtype: int64

In [6]:
category_map = {
    "ELECTRONICS": "Electronics",
    "Electronics & Accessories": "Electronics",
    "Electronic": "Electronics",
    "Electronicss": "Electronics"
}

In [7]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [8]:
df["category"].value_counts()

category
Electronics    121605
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [37]:
df["delivery_days"].unique()

array(['3', '1', '2', '4', '5', '6', '7', '-1', '15', '1-2 days', '0',
       'Same Day', 'Express'], dtype=object)

In [38]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [39]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [40]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [41]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [42]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [43]:
df["delivery_days"].unique()

array([ 3.,  1.,  2.,  4.,  5.,  6.,  7., nan, 15.,  0.])

In [44]:
df["delivery_days"].isnull().sum()

np.int64(718)

In [45]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [46]:
df["delivery_days"].describe()

count    121605.000000
mean          2.751688
std           1.710461
min           0.000000
25%           1.000000
50%           3.000000
75%           4.000000
max          15.000000
Name: delivery_days, dtype: float64

In [47]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [48]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [49]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
147,TXN_2024_00000148,2024-01-24,CUST_2017_00015387,PROD_001255,Samsung Galaxy S24 64GB Blue,Electronics,Smartphones,Samsung,91147.40,46.85,...,True,Republic Day Sale,4.5,Delivered,1,2024,1,0.19,True,3.7
239,TXN_2024_00000240,2024-01-21,CUST_2024_00013783,PROD_000038,Samsung Galaxy Note 5 32GB Black,Electronics,Smartphones,Samsung,114388.93,28.65,...,True,Republic Day Sale,5.0,Delivered,1,2024,1,0.20,True,3.3
251,TXN_2024_00000252,2024-01-14,CUST_2018_00013163,PROD_000990,Samsung Galaxy A53 64GB Black,Electronics,Smartphones,Samsung,25249.36,0.00,...,False,NaN,4.5,Delivered,1,2024,1,0.21,True,4.1
476,TXN_2024_00000477,2024-01-18,CUST_2024_00001609,PROD_000925,Vivo Y33s 256GB White,Electronics,Smartphones,Vivo,15845.53,0.00,...,False,NaN,4.0,Delivered,1,2024,1,0.18,True,3.7
633,TXN_2024_00000634,2024-01-24,CUST_2024_00006393,PROD_000299,Vivo V7+ 64GB Black,Electronics,Smartphones,Vivo,17561.00,33.40,...,True,Republic Day Sale,5.0,Delivered,1,2024,1,0.18,True,4.7


In [50]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [51]:
duplicates.shape

(1196, 34)

In [52]:
df.duplicated().sum()

np.int64(0)

In [53]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
90227,TXN_2024_00090228,2024-10-15,CUST_2015_00007253,PROD_001301,Xiaomi Mi 14 64GB Black,Electronics,Smartphones,Xiaomi,17418.08,19.16,...,True,Diwali Sale,4.0,Delivered,10,2024,4,0.19,True,4.2
121425,TXN_2024_00090228_DUP,2024-10-15,CUST_2015_00007253,PROD_001301,Xiaomi Mi 14 64GB Black,Electronics,Smartphones,Xiaomi,17418.08,19.16,...,True,Diwali Sale,4.0,Delivered,10,2024,4,0.19,True,4.2
4747,TXN_2024_00004748,2024-01-14,CUST_2015_00007494,PROD_000803,Apple iPhone 13 Pro 128GB Black,Electronics,Smartphones,Apple,244463.83,0.00,...,False,NaN,4.5,Delivered,1,2024,1,0.23,True,3.8
121124,TXN_2024_00004748_DUP,2024-01-14,CUST_2015_00007494,PROD_000803,Apple iPhone 13 Pro 128GB Black,Electronics,Smartphones,Apple,244463.83,0.00,...,False,NaN,4.5,Delivered,1,2024,1,0.23,True,3.8
79438,TXN_2024_00079439,2024-09-16,CUST_2015_00008398,PROD_000060,OnePlus OnePlus X 16GB White,Electronics,Smartphones,OnePlus,69358.76,0.00,...,False,NaN,4.5,Delivered,9,2024,3,0.24,True,3.3
121397,TXN_2024_00079439_DUP,2024-09-16,CUST_2015_00008398,PROD_000060,OnePlus OnePlus X 16GB White,Electronics,Smartphones,OnePlus,69358.76,0.00,...,False,NaN,4.5,Delivered,9,2024,3,0.24,True,3.3
83932,TXN_2024_00083933,2024-10-24,CUST_2016_00000552,PROD_000407,Xiaomi Redmi 5A 256GB Black,Electronics,Smartphones,Xiaomi,32605.92,22.36,...,True,Diwali Sale,4.0,Delivered,10,2024,4,0.23,True,4.6
121055,TXN_2024_00083933_DUP,2024-10-24,CUST_2016_00000552,PROD_000407,Xiaomi Redmi 5A 256GB Black,Electronics,Smartphones,Xiaomi,32605.92,22.36,...,True,Diwali Sale,4.0,Delivered,10,2024,4,0.23,True,4.6
112775,TXN_2024_00112776,2024-12-06,CUST_2016_00004671,PROD_001264,Samsung Galaxy S24+ 128GB Blue,Electronics,Smartphones,Samsung,121016.31,0.00,...,False,NaN,4.5,Delivered,12,2024,4,0.17,True,3.8
121354,TXN_2024_00112776_DUP,2024-12-06,CUST_2016_00004671,PROD_001264,Samsung Galaxy S24+ 128GB Blue,Electronics,Smartphones,Samsung,121016.31,0.00,...,False,NaN,4.5,Delivered,12,2024,4,0.17,True,3.8


In [54]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00007253  PROD_001301  2024-10-15  17418.08              2
CUST_2015_00007494  PROD_000803  2024-01-14  244463.83             2
CUST_2015_00008398  PROD_000060  2024-09-16  69358.76              2
CUST_2016_00000552  PROD_000407  2024-10-24  32605.92              2
CUST_2016_00004671  PROD_001264  2024-12-06  121016.31             2
CUST_2016_00011934  PROD_001251  2024-01-29  136341.52             2
CUST_2016_00017631  PROD_001948  2024-01-29  33878.20              2
CUST_2016_00018290  PROD_001929  2024-04-05  49436.31              2
CUST_2016_00019001  PROD_001942  2024-02-17  28311.73              2
CUST_2017_00007618  PROD_001789  2024-11-13  17199.47              2
dtype: int64

In [55]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [56]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [57]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2017_00015387,PROD_001255,2,2
1,CUST_2024_00013783,PROD_000038,1,2
2,CUST_2018_00013163,PROD_000990,1,2
3,CUST_2024_00001609,PROD_000925,1,2
4,CUST_2024_00006393,PROD_000299,1,2


In [58]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [59]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [60]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [61]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [62]:
df[df["transaction_id"]=="TXN_2015_00000280"]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [63]:

# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers ──────────────────────────────────
subcategory_caps = {
    "Smart Watch":        100000,
    "Tablets":            180000,
    "Smartphones":        400000,
    "Laptops":            550000,
    "TV & Entertainment": 500000,
    "Audio":              200000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: delivery_charges ──────────────────────────
df["delivery_charges"] = df["delivery_charges"].fillna(0)

# ── FIX 4: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = (df["subtotal_inr"] + df["delivery_charges"]).round(2)

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 300
Outliers fixed: 482
                         min        max      mean       50%
subcategory                                                
Audio                1073.98  154954.60  21040.10  22456.73
Laptops              5853.20  539678.80  91861.20  78277.78
Smart Watch          2041.30   73497.82  39554.13  38806.15
Smartphones          4061.14  365157.80  52516.44  32577.35
TV & Entertainment  10360.91  482720.90  87981.35  94456.71
Tablets              1817.25  179364.62  78262.07  79185.36

NaN in final_amount_inr:   0
Negative prices remaining: 0


In [64]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch: ✅ All within cap

Tablets: ✅ All within cap

Smartphones: ✅ All within cap

Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [65]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
UPI            67985
Credit Card    15837
COD            12165
Debit Card     11028
BNPL            7297
Net Banking     4926
Wallet          2367
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [66]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [67]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [68]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
66.29022598266602 MB


In [69]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 121605

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        84110       69.17
customer_rating      36680       30.16


In [70]:
df.to_csv("data_cleaning_2024.csv", index=False)
print("File saved successfully!")

File saved successfully!
